# TONTOUMA-BOT — Transcription en direct (parle → texte immédiat)

Un seul appel de fonction : tu parles, et la transcription des 4 modèles s'affiche immédiatement, avec le temps de traitement de chacun.

### Installation (une seule fois)

```bash
pip install jiwer psutil soundfile --break-system-packages
```


## 1. Configuration

In [ ]:
import time
import re
import unicodedata
import warnings
from pathlib import Path

import pandas as pd
import torch
from transformers import pipeline

warnings.filterwarnings("ignore")

# Using the models suggested by the user for Wolof ASR
MODELES_STT = {
    "M9and2M": "M9and2M/whisper-small-wolof", # cest u
    "Mon_LoRA": LOCAL_MODEL_DIR, # Use the locally downloaded model path
    "DVoice_Wav2Vec2": "speechbrain/asr-wav2vec2-dvoice-wolof"
}

SAMPLE_RATE = 16000
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE_INDEX = 0 if DEVICE == "cuda" else -1

AUDIO_DIR = Path("audios_test")
AUDIO_DIR.mkdir(exist_ok=True)

print(f"Device utilisé : {DEVICE.upper()}")

Device utilisé : CPU


## 2. Chargement des modèles Whisper (une seule fois)

In [3]:
print("Chargement des modèles STT...")
pipelines_stt = {}

for nom, chemin in MODELES_STT.items():
    t0 = time.perf_counter()
    try:
        pipelines_stt[nom] = pipeline("automatic-speech-recognition", model=chemin, device=DEVICE_INDEX)
        print(f" {nom:22s} chargé en {time.perf_counter()-t0:.1f}s")
    except Exception as e:
        print(f" {nom:22s} échec du chargement : {e}")

print(f"\n{len(pipelines_stt)}/{len(MODELES_STT)} modèles prêts.")


Chargement des modèles STT...


config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  967MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

...

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

 whisper-tiny           chargé en 5.0s

3/4 modèles prêts.


### Comment télécharger et charger un modèle localement

Pour 'cloner' un modèle depuis Hugging Face et le charger localement, vous devez d'abord le télécharger. Ensuite, le chemin dans `MODELES_STT` doit pointer vers le répertoire local où le modèle a été enregistré. La fonction `pipeline` reconnaîtra alors qu'il s'agit d'un chemin local.

In [ ]:
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor

LOCAL_MODEL_DIR = "local_models/whisper-small-wolof-downloaded"

print(f"Téléchargement du modèle M9and2M/whisper-small-wolof vers {LOCAL_MODEL_DIR}...")

# Créer le répertoire local si non existant
Path(LOCAL_MODEL_DIR).mkdir(parents=True, exist_ok=True)

# Télécharger le modèle et le tokenizer pour le sauvegarder localement
processor = AutoProcessor.from_pretrained("M9and2M/whisper-small-wolof")
model = AutoModelForSpeechSeq2Seq.from_pretrained("M9and2M/whisper-small-wolof")

# Sauvegarder localement
processor.save_pretrained(LOCAL_MODEL_DIR)
model.save_pretrained(LOCAL_MODEL_DIR)

print("Téléchargement et sauvegarde locale terminés.")

Téléchargement du modèle M9and2M/whisper-small-wolof vers local_models/whisper-small-wolof-downloaded...


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Téléchargement et sauvegarde locale terminés.


Maintenant que le modèle est téléchargé localement, vous pouvez mettre à jour l'entrée `whisper-lora-wolof` dans votre dictionnaire `MODELES_STT` pour qu'elle pointe vers ce chemin local. Cela devrait résoudre l'erreur de chemin.

In [ ]:
# Mettre à jour l'entrée 'whisper-lora-wolof' pour utiliser le modèle localement téléchargé
MODELES_STT["whisper-lora-wolof"] = LOCAL_MODEL_DIR

print("Dictionnaire MODELES_STT mis à jour. Vous pouvez maintenant relancer la cellule de chargement des modèles (section 2) pour charger le modèle local.")

# Afficher le dictionnaire mis à jour pour vérification
display(MODELES_STT)

#tokeniser un modele signifie le 


Dictionnaire MODELES_STT mis à jour. Vous pouvez maintenant relancer la cellule de chargement des modèles (section 2) pour charger le modèle local.


{'whisper-small': 'openai/whisper-small',
 'whisper-small-wolof': 'M9and2M/whisper-small-wolof',
 'whisper-lora-wolof': 'local_models/whisper-small-wolof-downloaded',
 'whisper-tiny': 'openai/whisper-tiny'}

## 3. Fonctions d'enregistrement + conversion audio

Exécute la cellule qui correspond à ton environnement (A pour Colab, B pour Jupyter local). Les deux définissent une fonction `_capturer_audio()` utilisée ensuite par `parler_et_transcrire()`.

Cette fonction prend en entrée un fichier audio en mémoire (format MP3, OGG, M4A, etc.) sous forme d'octets (bytes) et le convertit au format standard attendu par les modèles de machine learning :

In [ ]:
import subprocess
import io
import soundfile as sf
import numpy as np


def convertir_audio_16khz(contenu: bytes):
    processus = subprocess.run(
        ["ffmpeg", "-y", "-i", "pipe:0", "-f", "wav", "-ac", "1", "-ar", str(SAMPLE_RATE), "pipe:1"],
        input=contenu, stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True,
    )
    audio, sr = sf.read(io.BytesIO(processus.stdout), dtype="float32")
    return audio, sr



# Cette fonction prend le tableau NumPy généré précédemment et l'enregistre physiquement sur le 
# disque dur au format WAV PCM 16-bit (subtype="PCM_16"), le format audio standard lisible par n'importe quel lecteur.
def sauvegarder_wav(audio, sr, chemin):
    sf.write(chemin, audio, sr, subtype="PCM_16")


In [ ]:
# cette fonction est utilisée pour capturer l'audio depuis le micro de l'utilisateur dans un environnement Google Colab.
#  Elle utilise JavaScript pour accéder au micro, enregistre l'audio, puis le convertit en un format compatible avec les modèles STT.


try:
    from google.colab import output
    from base64 import b64decode
    from IPython.display import Javascript, display

    RECORD_JS = """
    const b2text = blob => new Promise(resolve => {
      const reader = new FileReader();
      reader.onloadend = () => resolve(reader.result);
      reader.readAsDataURL(blob);
    });

    var recorder;
    async function recordAudio() {
      const stream = await navigator.mediaDevices.getUserMedia({audio: true});
      recorder = new MediaRecorder(stream);
      let chunks = [];
      recorder.ondataavailable = e => chunks.push(e.data);
      recorder.start();

      await new Promise(resolve => {
        const btn = document.createElement('button');
        btn.textContent = '⏹ Arrêter';
        btn.style = 'font-size:16px;padding:10px;background:#d9534f;color:white;border:none;border-radius:5px;cursor:pointer;';
        document.body.appendChild(btn);
        btn.onclick = () => { recorder.stop(); document.body.removeChild(btn); resolve(); };
      });

      await new Promise(resolve => recorder.onstop = resolve);
      const blob = new Blob(chunks);
      const b64 = await b2text(blob);
      return b64;
    }
    """

    def _capturer_audio(nom_fichier: str, duree_s: float = None):
        display(Javascript(RECORD_JS))
        print("🎙️  Clique sur le bouton rouge pour démarrer, puis à nouveau pour arrêter...")
        data_url = output.eval_js("recordAudio()")
        _, encoded = data_url.split(",", 1)
        contenu = b64decode(encoded)
        audio, sr = convertir_audio_16khz(contenu)
        chemin = AUDIO_DIR / f"{nom_fichier}.wav"
        sauvegarder_wav(audio, sr, str(chemin))
        return str(chemin)

    print("✅ Colab détecté — capture prête.")

except ImportError:
    print("Pas sur Colab — utilise la cellule B (sounddevice) ci-dessous.")


✅ Colab détecté — capture prête.


In [ ]:
# Cellule B - Jupyter local (hors Colab)
# ici on utilise la bibliothèque sounddevice pour capturer l'audio depuis le micro de l'utilisateur.
# puis on le convertit en un format compatible avec les modèles STT.

try:
    import sounddevice as sd

    def _capturer_audio(nom_fichier: str, duree_s: float = 5.0):
        print(f"🎙️  Enregistrement de {duree_s}s dans 2 secondes...")
        time.sleep(2)
        print("🔴 Parle maintenant...")
        audio = sd.rec(int(duree_s * SAMPLE_RATE), samplerate=SAMPLE_RATE, channels=1, dtype="float32")
        sd.wait()
        print("⏹ Terminé.")
        chemin = AUDIO_DIR / f"{nom_fichier}.wav"
        sauvegarder_wav(audio.flatten(), SAMPLE_RATE, str(chemin))
        return str(chemin)

    print("✅ sounddevice détecté — capture prête.")

except ImportError:
    print("⚠️  pip install sounddevice soundfile --break-system-packages")


⚠️  pip install sounddevice soundfile --break-system-packages


## 4. 🎙️ Parle → transcription immédiate par les 4 modèles

**Un seul appel** : enregistrement + transcription automatique par tous les modèles chargés, avec affichage direct des résultats.

In [ ]:
# ici on définit la fonction principale qui enregistre la voix de l'utilisateur et transcrit immédiatement avec tous
#  les modèles STT chargés. Elle prend en entrée le nom du fichier audio à sauvegarder et la durée d'enregistrement en 
# secondes. Si aucun nom de fichier n'est fourni, un nom est généré automatiquement. La fonction retourne un tableau 
# comparatif des transcriptions par modèle.

def parler_et_transcrire(nom_fichier: str = None, duree_s: float = 5.0):
    """
    Enregistre ta voix et transcrit immédiatement avec tous les modèles STT chargés.

    Arguments:
    nom_fichier -- nom du fichier audio à sauvegarder (auto-généré si None)
    duree_s     -- durée d'enregistrement en secondes (Jupyter local uniquement, ignoré sur Colab)

    Retourne:
    df -- tableau comparatif des transcriptions par modèle
    """
    if nom_fichier is None:
        nom_fichier = f"live_{int(time.time())}"

    # --- Capture audio ---
    chemin_audio = _capturer_audio(nom_fichier, duree_s)

    # --- Transcription immédiate par tous les modèles ---
    print(f"\n Transcription en cours ({len(pipelines_stt)} modèles)...\n")
    resultats = []

    for nom_modele, modele in pipelines_stt.items():
        t0 = time.perf_counter()
        try:
            # Remove 'language' from generate_kwargs as it causes 'Unsupported language' error.
            # For Wav2Vec2 models, remove generate_kwargs entirely as it might not be applicable.
            if "wav2vec2" in nom_modele.lower():
                sortie = modele(chemin_audio)
            else:
                sortie = modele(chemin_audio, generate_kwargs={"task": "transcribe"})
            temps_ms = (time.perf_counter() - t0) * 1000
            prediction = sortie["text"].strip()
            echec = False
        except Exception as e:
            prediction = f"[ERREUR: {e}]"
            temps_ms = (time.perf_counter() - t0) * 1000
            echec = True

        resultats.append({
            "modele": nom_modele,
            "transcription": prediction,
            "temps_ms": round(temps_ms, 1),
            "echec": echec,
        })
        statut = "❌" if echec else "✅"
        print(f"  {statut} {nom_modele:22s} ({temps_ms:6.0f} ms) : {prediction}")

    df = pd.DataFrame(resultats)
    return df


# Utilisation directe :
df_live = parler_et_transcrire()

<IPython.core.display.Javascript object>

🎙️  Clique sur le bouton rouge pour démarrer, puis à nouveau pour arrêter...

🧠 Transcription en cours (3 modèles)...



[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.
[

  ✅ whisper-small          (172372 ms) : ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ
  ✅ whisper-small-wolof    (167645 ms) : damaa bëgga dem jangi dem ekool am kayitu juddu
  ✅ whisper-tiny           ( 35400 ms) : Kemudian, bagaimana? Kemudian, bagaimana? Kemudian, bagaimana? Kemudian, bagaimana? Kemudian, bagaimana? Kemudian, bagaimana? Kemudian, bagaimana? Kemudian, bagaimana? Kemudian, bagaimana? Kemudian, bagaimana? Kemudian, bagaimana? Kemudian, bagaimana? Kemudian, bagaimana? Kemudian, bagaimana? Kemudian, bagaimana? Kemudian, bagaimana? Kemudian, bagaimana? Kemudian, bagaimana? Kemudian, bagaimana? Kemudian, bagaimana? Kemudian, bagaimana? Kemudian, bagaimana? Kemudian, bagaimana? Kemudian, bagaimana? Kemudian, bagaimana? Kemudian

## 5. (Optionnel) Comparer au texte réellement dit

Si tu veux mesurer le WER/CER, tape ce que tu as vraiment dit juste après l'appel ci-dessus.

In [ ]:
damaa bëgga dem jangi dem ekool am kayitu juddu
try:
    from jiwer import wer as jiwer_wer, cer as jiwer_cer
    JIWER_OK = True
except ImportError:
    JIWER_OK = False
    print("⚠️  jiwer non installé -> pip install jiwer")


def normaliser_texte(texte: str) -> str:
    texte = str(texte).lower().strip()
    texte = unicodedata.normalize("NFKC", texte)
    texte = re.sub(r"[^\w\sàâäéèêëîïôöùûüÿñç]", " ", texte)
    texte = re.sub(r"\s+", " ", texte)
    return texte


def evaluer_derniere_transcription(df, reference: str = None):
    if reference is None:
        reference = input("Qu'as-tu VRAIMENT dit : ")

    if not JIWER_OK:
        print("jiwer non installé, impossible de calculer WER/CER.")
        return df

    ref_norm = normaliser_texte(reference)
    df = df.copy()
    df["wer"] = df["transcription"].apply(lambda p: round(jiwer_wer(ref_norm, normaliser_texte(p)), 3))
    df["cer"] = df["transcription"].apply(lambda p: round(jiwer_cer(ref_norm, normaliser_texte(p)), 3))

    display(df[["modele", "transcription", "wer", "cer", "temps_ms"]])
    meilleur = df.loc[df["wer"].idxmin(), "modele"]
    print(f"\n🏆 Meilleure transcription : {meilleur}")
    return df


# Utilisation :
df_live = evaluer_derniere_transcription(df_live)

# le WER mesure la proportion de mots incorrects dans la transcription par rapport à la référence.
#  Un WER de 0 signifie une transcription parfaite, tandis qu'un WER de 1 signifie que tous les mots sont incorrects.
#  Le CER (Character Error Rate) mesure la proportion de caractères incorrects. Ces métriques sont utiles pour évaluer
#  la performance des modèles STT sur des phrases spécifiques.



Qu'as-tu VRAIMENT dit : damaa bëgga dem jangi dem ekool am kayitu juddu


,modele,transcription,wer,cer,temps_ms
0,whisper-small,ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ლ ...,16.444,6.106,172372.4
1,whisper-small-wolof,damaa bëgga dem jangi dem ekool am kayitu juddu,0.000,0.000,167645.1
2,whisper-tiny,"Kemudian, bagaimana? Kemudian, bagaimana? Kemu...",12.333,21.574,35399.7



🏆 Meilleure transcription : whisper-small-wolof


In [ ]:
pip install jiwer

#jiwer est une bibliothèque Python qui fournit des fonctions pour calculer le Word Error Rate (WER) et 
# le Character Error Rate (CER), deux métriques couramment utilisées pour évaluer la performance des systèmes de 
# reconnaissance vocale.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 20.8 MB/s eta 0:00:00


In [23]:
meilleur_modele = df_live.loc[df_live['wer'].idxmin()]

print("Meilleur modèle :", meilleur_modele["modele"])
print("WER :", meilleur_modele["wer"])

Meilleur modèle : whisper-small-wolof
WER : 0.0


In [25]:
import json
import os
import shutil

os.makedirs("./models", exist_ok=True)

with open("./models/meilleur_asr.json", "w", encoding="utf-8") as fichier:
    json.dump(meilleur_modele.to_dict(), fichier, ensure_ascii=False, indent=2)

print("Modèle sélectionné :", meilleur_modele["modele"])

Modèle sélectionné : whisper-small-wolof


In [ ]:
#sauvegarder le meilleur modèle dans un répertoire spécifique
meilleur_modele_nom = meilleur_modele["modele"]

